# Báo Cáo Thực Nghiệm: Mạng Tích Chập 1D (1D CNN) Hồi Quy Bằng TensorFlow / Keras
## Bài Toán: Dự Đoán Giá Nhà (Real Estate Continuous Price Regression)
**Học phần:** Phát triển Hệ thống Thông minh (Intelligent System Development)  
**Nhiệm vụ:** Assignment 04 - Ánh xạ kiến trúc 1D CNN Hồi quy sang TensorFlow (`layers.Conv1D`, Linear Output, MSE loss)  
**Dữ liệu:** `usa_real_estate_150k.csv` (150,000 giao dịch)


In [1]:
import os
os.environ["MPLBACKEND"] = "Agg"
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses
import numpy as np
import pandas as pd
import time
import json
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras import layers, models, optimizers, losses

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Phiên bản TensorFlow: {tf.__version__}")


Phiên bản TensorFlow: 2.21.0


### 1. Đọc & Tiền Xử Lý Dữ Liệu Đồng Bộ


In [2]:
DATA_PATH = "../../../Assigment 02/house_price/data/usa_real_estate_150k.csv"
if not os.path.exists(DATA_PATH):
    DATA_PATH = "src/Assigment 02/house_price/data/usa_real_estate_150k.csv"

df_raw = pd.read_csv(DATA_PATH)
df_clean = df_raw.dropna(subset=['price', 'house_size', 'bed', 'bath']).copy()
df_clean = df_clean[(df_clean['price'] >= 10000) & (df_clean['price'] <= 5000000)]
df_clean = df_clean[(df_clean['house_size'] >= 200) & (df_clean['house_size'] <= 20000)]

df_clean['log_house_size'] = np.log(df_clean['house_size'])
df_clean['total_rooms'] = df_clean['bed'] + df_clean['bath']
df_clean['bed_bath_prod'] = df_clean['bed'] * df_clean['bath']
df_clean['sqft_per_room'] = df_clean['house_size'] / (df_clean['total_rooms'] + 1)
df_clean['bath_bed_ratio'] = df_clean['bath'] / (df_clean['bed'] + 1)
df_clean['log_acre_lot'] = np.log1p(df_clean['acre_lot'].fillna(df_clean['acre_lot'].median()))
df_clean['log_price'] = np.log(df_clean['price'])

features = ['log_house_size', 'bed', 'bath', 'total_rooms', 'bed_bath_prod', 'sqft_per_room', 'bath_bed_ratio', 'log_acre_lot']

X_data = df_clean[features].values.astype(np.float32)
y_data = df_clean['log_price'].values.astype(np.float32)

X_train_raw, X_test_raw, y_train_log, y_test_log = train_test_split(
    X_data, y_data, test_size=0.20, random_state=RANDOM_SEED
)

scaler_x = StandardScaler()
X_train_scaled = scaler_x.fit_transform(X_train_raw)
X_test_scaled = scaler_x.transform(X_test_raw)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train_log.reshape(-1, 1)).ravel()
y_test_scaled = scaler_y.transform(y_test_log.reshape(-1, 1)).ravel()

X_train_tf = X_train_scaled[:, :, np.newaxis]
X_test_tf = X_test_scaled[:, :, np.newaxis]

print(f"X_train_tf: {X_train_tf.shape}, y_train_scaled: {y_train_scaled.shape}")


X_train_tf: (120000, 8, 1), y_train_scaled: (120000,)


### 2. Định Nghĩa Mô Hình 1D CNN Hồi Quy Bằng TensorFlow
- `layers.Conv1D(16, kernel_size=3, activation='relu')`
- `layers.Conv1D(32, kernel_size=3, activation='relu')`
- `layers.MaxPooling1D(pool_size=2)`
- `layers.Flatten()`
- `layers.Dense(16, activation='relu')`
- `layers.Dense(1, activation=None)` (Nơ-ron tuyến tính dự đoán giá trị liên tục)


In [3]:
model = models.Sequential([
    layers.Input(shape=(8, 1)),
    layers.Conv1D(filters=16, kernel_size=3, activation='relu'),
    layers.Conv1D(filters=32, kernel_size=3, activation='relu'),
    layers.MaxPooling1D(pool_size=2),
    layers.Flatten(),
    layers.Dense(16, activation='relu'),
    layers.Dense(1, activation=None) # Linear output
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.002),
    loss='mean_squared_error',
    metrics=['mae']
)

model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 6, 16)          │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 4, 32)          │         1,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 2, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,689 (10.50 KB)

 Trainable params: 2,689 (10.50 KB)

 Non-trainable params: 0 (0.00 B)

### 3. Huấn Luyện Mô Hình Hồi Quy


In [4]:
epochs = 12
batch_size = 256

start_time = time.time()
print("Bắt đầu huấn luyện mô hình 1D CNN Hồi quy TensorFlow...")

fit_history = model.fit(
    X_train_tf, y_train_scaled,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(X_test_tf, y_test_scaled),
    verbose=1
)

train_time = time.time() - start_time
print(f"\nHuấn luyện TensorFlow hoàn tất trong: {train_time:.2f} giây!")


Bắt đầu huấn luyện mô hình 1D CNN Hồi quy TensorFlow...
Epoch 1/12


  1/469 ━━━━━━━━━━━━━━━━━━━━ 5:43 735ms/step - loss: 0.9275 - mae: 0.7640

 43/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.7381 - mae: 0.6458    

 88/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6854 - mae: 0.6267

132/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6618 - mae: 0.6154

175/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.6549 - mae: 0.6122

... (155 outputs left out)

420/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5931 - mae: 0.5844

462/469 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 0.5935 - mae: 0.5847

469/469 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - loss: 0.5936 - mae: 0.5849 - val_loss: 0.5974 - val_mae: 0.5855



Huấn luyện TensorFlow hoàn tất trong: 8.77 giây!


### 4. Đánh Giá Toàn Diện & Trực Quan Hóa Kết Quả


In [5]:
val_preds_scaled = model.predict(X_test_tf).ravel()
val_preds_log = scaler_y.inverse_transform(val_preds_scaled.reshape(-1, 1)).ravel()

r2_val = r2_score(y_test_log, val_preds_log)
rmse_log = np.sqrt(mean_squared_error(y_test_log, val_preds_log))
mae_log = mean_absolute_error(y_test_log, val_preds_log)

y_test_dollars = np.exp(y_test_log)
preds_dollars = np.exp(val_preds_log)
mae_dollars = mean_absolute_error(y_test_dollars, preds_dollars)

print(f"=== KẾT QUẢ ĐÁNH GIÁ 1D CNN HỒI QUY (TENSORFLOW) ===")
print(f"• R² Score (Log space)  : {r2_val:.4f}")
print(f"• RMSE (Log space)      : {rmse_log:.4f}")
print(f"• MAE (Log space)       : {mae_log:.4f}")
print(f"• MAE (Thực tế USD)     : ${mae_dollars:,.2f}")
print(f"• Thời gian huấn luyện  : {train_time:.2f} giây")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(range(1, epochs + 1), fit_history.history["loss"], 'o-', color='#f59e0b', label='Train MSE')
axes[0].plot(range(1, epochs + 1), fit_history.history["val_loss"], 's--', color='#ef4444', label='Val MSE')
axes[0].set_title('TensorFlow: Đường Cong Hàm Mất Mát MSE Theo Epoch', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean Squared Error')
axes[0].legend()

sample_idx = np.random.choice(len(y_test_log), size=3000, replace=False)
axes[1].scatter(np.asarray(y_test_log)[sample_idx], val_preds_log[sample_idx], alpha=0.3, color='#d97706', s=15)
min_val = min(y_test_log.min(), val_preds_log.min())
max_val = max(y_test_log.max(), val_preds_log.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Đường lý tưởng (y = ŷ)')
axes[1].set_title('TensorFlow: Tương Quan Giá Thực Tế vs Giá Dự Đoán (Log)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Giá Thực Tế (log)')
axes[1].set_ylabel('Giá Dự Đoán (log)')
axes[1].legend()

plt.tight_layout()
os.makedirs("../reports/figures", exist_ok=True)
plt.savefig("../reports/figures/03_house_price_tensorflow_results.png", dpi=300)
plt.show()


  1/938 ━━━━━━━━━━━━━━━━━━━━ 36s 39ms/step

173/938 ━━━━━━━━━━━━━━━━━━━━ 0s 291us/step

358/938 ━━━━━━━━━━━━━━━━━━━━ 0s 281us/step

542/938 ━━━━━━━━━━━━━━━━━━━━ 0s 278us/step

727/938 ━━━━━━━━━━━━━━━━━━━━ 0s 276us/step

913/938 ━━━━━━━━━━━━━━━━━━━━ 0s 275us/step

938/938 ━━━━━━━━━━━━━━━━━━━━ 0s 318us/step


=== KẾT QUẢ ĐÁNH GIÁ 1D CNN HỒI QUY (TENSORFLOW) ===
• R² Score (Log space)  : 0.4036
• RMSE (Log space)      : 0.6820
• MAE (Log space)       : 0.5166
• MAE (Thực tế USD)     : $289,379.53
• Thời gian huấn luyện  : 8.77 giây


/var/folders/x0/279_kvcj3dvbd1fkpvt_8j7r0000gn/T/ipykernel_3469/1555278981.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5. Lưu Trữ Kết Quả Thực Nghiệm & Tổng Hợp Đối Chuẩn


In [6]:
results_df = pd.DataFrame([{
    "Framework": "TensorFlow",
    "Dataset": "House Price (150k)",
    "R2_Score": r2_val,
    "RMSE_Log": rmse_log,
    "MAE_Log": mae_log,
    "MAE_Dollars": mae_dollars,
    "Train_Time_Sec": train_time,
    "Epochs": epochs,
    "Batch_Size": batch_size
}])

results_df.to_csv("../reports/model_results_tensorflow.csv", index=False)
print("Đã lưu kết quả thành công vào ../reports/model_results_tensorflow.csv")

dfs = []
for fname in ["model_results_numpy.csv", "model_results_pytorch.csv", "model_results_tensorflow.csv"]:
    p = os.path.join("../reports", fname)
    if os.path.exists(p):
        dfs.append(pd.read_csv(p))

if dfs:
    summary_df = pd.concat(dfs, ignore_index=True)
    summary_df.to_csv("../reports/model_results.csv", index=False)
    print("\n=== BẢNG TỔNG HỢP ĐỐI CHUẨN 3 FRAMEWORK TRÊN TẬP HOUSE PRICE ===")
    display(summary_df)


Đã lưu kết quả thành công vào ../reports/model_results_tensorflow.csv

=== BẢNG TỔNG HỢP ĐỐI CHUẨN 3 FRAMEWORK TRÊN TẬP HOUSE PRICE ===


,Framework,Dataset,R2_Score,RMSE_Log,MAE_Log,MAE_Dollars,Train_Time_Sec,Epochs,Batch_Size
0,NumPy From Scratch,House Price (150k),0.390118,0.689680,0.520435,290959.572268,197.148848,12,256
1,PyTorch,House Price (150k),0.399917,0.684117,0.518442,290404.468750,41.479366,12,256
2,TensorFlow,House Price (150k),0.403553,0.682041,0.516647,289379.531250,8.769331,12,256
